# Linger Workflow
Below is the file structure. We start with fresh `LINGER_output/` and `data/` folders.
```
.
├── LINGER_data/
│   └── data_bulk/
│
├── LINGER_output/
│   ├── cell_population_TF_RE_binding.txt
│   ├── cell_population_cis_regulatory.txt
│   ├── cell_population_trans_regulatory.txt
│   └── ...
│
└── data/
    ├── Peaks.txt
    ├── TG_pseudobulk.tsv
    ├── RE_pseudobulk.tsv
    ├── adata_RNA.h5ad
    └── adata_ATAC.h5ad
```

In [2]:
rm -rf LINGER_output/*

In [3]:
rm -rf data/*

## 1. Get the input data
- `.h5` matrix with both RNA and ATAC
- cell type annotation

In [4]:
%%bash
wget --progress=bar:force -O data/pbmc_granulocyte_sorted_10k_filtered_feature_bc_matrix.h5 https://cf.10xgenomics.com/samples/cell-arc/1.0.0/pbmc_granulocyte_sorted_10k/pbmc_granulocyte_sorted_10k_filtered_feature_bc_matrix.h5
wget --progress=bar:force -O data --load-cookies /tmp/cookies.txt "https://docs.google.com/uc?export=download&confirm=$(wget --quiet --save-cookies /tmp/cookies.txt --keep-session-cookies --no-check-certificate 'https://docs.google.com/uc?export=download&id=17PXkQJr8fk0h90dCkTi3RGPmFNtDqHO_' -O- | sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\1\n/p')&id=17PXkQJr8fk0h90dCkTi3RGPmFNtDqHO_" -O data/PBMC_label.txt && rm -rf /tmp/cookies.txt

--2026-03-06 11:11:23--  https://cf.10xgenomics.com/samples/cell-arc/1.0.0/pbmc_granulocyte_sorted_10k/pbmc_granulocyte_sorted_10k_filtered_feature_bc_matrix.h5
Resolving cf.10xgenomics.com (cf.10xgenomics.com)... 104.18.1.173, 104.18.0.173, 2606:4700::6812:ad, ...
Connecting to cf.10xgenomics.com (cf.10xgenomics.com)|104.18.1.173|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 162282142 (155M) [binary/octet-stream]
Saving to: 'data/pbmc_granulocyte_sorted_10k_filtered_feature_bc_matrix.h5'

data/pbmc_granulocy 100%[===================>] 154.76M  22.4MB/s    in 7.3s    

2026-03-06 11:11:32 (21.2 MB/s) - 'data/pbmc_granulocyte_sorted_10k_filtered_feature_bc_matrix.h5' saved [162282142/162282142]

--2026-03-06 11:11:33--  https://docs.google.com/uc?export=download&confirm=&id=17PXkQJr8fk0h90dCkTi3RGPmFNtDqHO_
Resolving docs.google.com (docs.google.com)... 173.194.76.100, 173.194.76.102, 173.194.76.113, ...
Connecting to docs.google.com (docs.google.com)|173.194

In [5]:
ls data

PBMC_label.txt  pbmc_granulocyte_sorted_10k_filtered_feature_bc_matrix.h5


In [ ]:
import scanpy as sc
adata = sc.read_10x_h5('data/pbmc_granulocyte_sorted_10k_filtered_feature_bc_matrix.h5', gex_only=False)

In [ ]:
import scipy.sparse as sp
import pandas as pd
matrix=adata.X.T
adata.var['gene_ids']=adata.var.index
features=pd.DataFrame(adata.var['gene_ids'].values.tolist(),columns=[1])
features[2]=adata.var['feature_types'].values
barcodes=pd.DataFrame(adata.obs_names,columns=[0])
label = pd.read_csv('data/PBMC_label.txt',sep='\t',header=0)
from LingerGRN.preprocess import *
adata_RNA,adata_ATAC=get_adata(matrix,features,barcodes,label)# adata_RNA and adata_ATAC are scRNA and scATAC

In [8]:
adata_RNA

View of AnnData object with n_obs × n_vars = 9543 × 36601
    obs: 'barcode', 'sample', 'label', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt'
    var: 'gene_ids', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'

In [9]:
adata_ATAC

View of AnnData object with n_obs × n_vars = 9543 × 108377
    obs: 'barcode', 'sample', 'label'
    var: 'gene_ids'

## 2. Filter cells and genes

In [ ]:
sc.pp.filter_cells(adata_RNA, min_genes=200)
sc.pp.filter_genes(adata_RNA, min_cells=3)
sc.pp.filter_cells(adata_ATAC, min_genes=200)
sc.pp.filter_genes(adata_ATAC, min_cells=3)

selected_barcode = list(set(adata_RNA.obs['barcode'].values) & set(adata_ATAC.obs['barcode'].values))

barcode_idx = pd.DataFrame(range(adata_RNA.shape[0]), index=adata_RNA.obs['barcode'].values)
adata_RNA = adata_RNA[barcode_idx.loc[selected_barcode][0]]

barcode_idx = pd.DataFrame(range(adata_ATAC.shape[0]), index=adata_ATAC.obs['barcode'].values)
adata_ATAC = adata_ATAC[barcode_idx.loc[selected_barcode][0]]

## 3 Pseudo-bulk (`pseudo_bulk.py`)
This function creates *metacells* : aggregated pseudo-cells that represent local neighborhoods in the data. 

In [ ]:
# Generate pseudo-bulk/metacell
import os
from LingerGRN.pseudo_bulk import *

samplelist=list(set(adata_ATAC.obs['sample'].values)) # sample is generated from cell barcode 
tempsample=samplelist[0]

TG_pseudobulk=pd.DataFrame([])
RE_pseudobulk=pd.DataFrame([])

n_samples = adata_RNA.obs['sample'].nunique()
singlepseudobulk = (n_samples > 10)

for tempsample in samplelist:

    # get cells from only tempsample
    adata_RNAtemp = adata_RNA[adata_RNA.obs['sample' ] == tempsample]
    adata_ATACtemp = adata_ATAC[adata_ATAC.obs['sample'] == tempsample]

    TG_pseudobulk_temp, RE_pseudobulk_temp = pseudo_bulk(adata_RNAtemp, adata_ATACtemp, singlepseudobulk)  
    
    TG_pseudobulk = pd.concat([TG_pseudobulk, TG_pseudobulk_temp], axis=1)
    RE_pseudobulk = pd.concat([RE_pseudobulk, RE_pseudobulk_temp], axis=1)
    
    RE_pseudobulk[RE_pseudobulk > 100] = 100

In [ ]:
if not os.path.exists('data/'):
    os.mkdir('data/')
    
adata_ATAC.write('data/adata_ATAC.h5ad')
adata_RNA.write('data/adata_RNA.h5ad')
TG_pseudobulk=TG_pseudobulk.fillna(0)
RE_pseudobulk=RE_pseudobulk.fillna(0)
pd.DataFrame(adata_ATAC.var['gene_ids']).to_csv('data/Peaks.txt',header=None,index=None)
TG_pseudobulk.to_csv('data/TG_pseudobulk.tsv')
RE_pseudobulk.to_csv('data/RE_pseudobulk.tsv')

In [13]:
ls data

PBMC_label.txt     adata_ATAC.h5ad
Peaks.txt          adata_RNA.h5ad
RE_pseudobulk.tsv  pbmc_granulocyte_sorted_10k_filtered_feature_bc_matrix.h5
TG_pseudobulk.tsv


## 4. Preprocess

In [14]:
import os
from LingerGRN.preprocess import *
source = "/globalsc/ucl/inma/vangysel/Linger/"
GRNdir =  source + "LINGER_data/data_bulk/"
genome = 'hg38'
outdir = source + "LINGER_output/" 

In [15]:
method = 'baseline'  

In [16]:
preprocess(TG_pseudobulk, RE_pseudobulk, GRNdir, genome, method, outdir)

Overlap the regions with bulk data ...


### About the preprocess function
Calls `extract_overlap_regions(genome, GRNdir, outdir, method=baseline)` 
 - input:
   - `data/Peaks.txt` (list of our peaks `chr:start-end`)
 - output:
   - `LINGER_output/Region.bed` : ATAC peaks from `Peaks.txt` reformatted as a 3-column BED file (`chr`, `start`, `end`)
   - `LINGER_data/data_bulk/Region_overlap_{chr}.bed` : Intersection of our peaks with the reference hg38 peaks for that chromosome — tells LINGER which of our peaks overlap with the bulk atlas peaks

In [29]:
import pandas as pd
peaks = pd.read_csv("data/Peaks.txt")
peaks.shape

(107207, 1)

In [30]:
peaks.head(2)

,chr1:10109-10357
0,chr1:180730-181630
1,chr1:191491-191736


In [35]:
regions = pd.read_csv("LINGER_output/Region.bed", sep="\t")
regions.shape

(107173, 3)

In [36]:
regions.head(2)

,chr1,10109,10357
0,chr1,180730,181630
1,chr1,191491,191736


In [39]:
ls LINGER_output

Region.bed                Region_overlap_chr16.bed  Region_overlap_chr3.bed
Region_overlap_chr1.bed   Region_overlap_chr17.bed  Region_overlap_chr4.bed
Region_overlap_chr10.bed  Region_overlap_chr18.bed  Region_overlap_chr5.bed
Region_overlap_chr11.bed  Region_overlap_chr19.bed  Region_overlap_chr6.bed
Region_overlap_chr12.bed  Region_overlap_chr2.bed   Region_overlap_chr7.bed
Region_overlap_chr13.bed  Region_overlap_chr20.bed  Region_overlap_chr8.bed
Region_overlap_chr14.bed  Region_overlap_chr21.bed  Region_overlap_chr9.bed
Region_overlap_chr15.bed  Region_overlap_chr22.bed  Region_overlap_chrX.bed


## 5. Training
In baseline mode, there is no training so `LINGER_tr.training` does nothing. 
In `training` function there is only two branches :
- `if method=='LINGER'`
- `if method=='scNN'`

In [37]:
import LingerGRN.LINGER_tr as LINGER_tr
LINGER_tr.training(GRNdir,method,outdir,'ReLU','Human')

## 6. Cell population GRN

| Method                             | Files                                                       |
| ---------------------------------- | ----------------------------------------------------------- |
| `TF_RE_binding(method='baseline')` | `Primary_TF_RE_{chr}.txt`, `Region_overlap`                 |
| `cis_reg(method='baseline')`       | `Primary_RE_TG_{chr}.txt`, `RE_TG_distance_{chr}.txt`       |
| `trans_reg(method='baseline')`     | `cell_population_TF_RE_binding.txt`, cis-regulatory results |

<br>

### 6.1 TF-RE

In [44]:
import LingerGRN.LL_net as LL_net
LL_net.TF_RE_binding(GRNdir,adata_RNA,adata_ATAC,genome,method,outdir)

Generating cellular population TF binding strength ...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 23/23 [00:41<00:00,  1.80s/it]


It maps TF-RE binding scores from the bulk atlas onto your dataset's peaks.<br>
For each chromosome it finds which of your ATAC peaks overlap with the reference atlas peaks, pulls the precomputed TF binding scores for those reference peaks, and filters to only the TFs that are actually expressed in your RNA data. <br>
The result is a matrix of peaks × TFs giving a binding score for each TF at each peak in your dataset.

```
TF_RE_binding(method='baseline')
  └── for each chr:
        TF_RE_binding_chr(...)
          ├── load_region(Region_overlap_{chr}.bed)
          │     → O_overlap, N_overlap, O_overlap_hg19_u
          ├── load_TF_RE(Primary_TF_RE_{chr}.txt)
          │     → TF-RE prior matrix mapped to your peaks
          └── filter by TFs present in adata_RNA.var
                → chr{N}_cell_population_TF_RE_binding.txt
  └── concat all chrs
        → cell_population_TF_RE_binding.txt  (peaks × TFs)
```
<br>

| Variable | Content |
|---|---|
| `O_overlap` | Reference hg38 peak for each overlap (one entry per overlap) |
| `N_overlap` | Our peak for each overlap (paired with `O_overlap`) |
| `O_overlap_u` | Unique reference hg38 peaks |
| `N_overlap_u` | Unique user peaks |
| `O_overlap_hg19_u` | Unique reference peaks translated to hg19 coordinates (used to index prior knowledge files) |
| `Primary_TF_RE_{chr}.txt`| precomputed TF-to-RE binding matrix from the bulk atlas, indexed in hg19 coordinates

In [45]:
ls LINGER_output

Region.bed                cell_population_TF_RE_binding.txt
Region_overlap_chr1.bed   chr10_cell_population_TF_RE_binding.txt
Region_overlap_chr10.bed  chr11_cell_population_TF_RE_binding.txt
Region_overlap_chr11.bed  chr12_cell_population_TF_RE_binding.txt
Region_overlap_chr12.bed  chr13_cell_population_TF_RE_binding.txt
Region_overlap_chr13.bed  chr14_cell_population_TF_RE_binding.txt
Region_overlap_chr14.bed  chr15_cell_population_TF_RE_binding.txt
Region_overlap_chr15.bed  chr16_cell_population_TF_RE_binding.txt
Region_overlap_chr16.bed  chr17_cell_population_TF_RE_binding.txt
Region_overlap_chr17.bed  chr18_cell_population_TF_RE_binding.txt
Region_overlap_chr18.bed  chr19_cell_population_TF_RE_binding.txt
Region_overlap_chr19.bed  chr1_cell_population_TF_RE_binding.txt
Region_overlap_chr2.bed   chr20_cell_population_TF_RE_binding.txt
Region_overlap_chr20.bed  chr21_cell_population_TF_RE_binding.txt
Region_overlap_chr21.bed  chr22_cell_population_TF_RE_binding.txt
Region_overlap_ch

In [49]:
tf_re = pd.read_csv("LINGER_output/cell_population_TF_RE_binding.txt", sep="\t")
tf_re.shape

(97042, 452)

In [51]:
tf_re.head(2)

,Unnamed: 0,PGR,BCL6,E2F4,MLXIP,XBP1,KLF1,CLOCK,MAF,FOXK2,...,ZBTB6,LMX1B,E2F5,GLIS1,IRF1,NRL,CTCF,HOMEZ,NPAS2,PLAGL1
0,chr1:100028489-100029404,0.430152,0.318511,0.599682,0.520004,0.34289,0.204503,0.0,0.0,0.0,...,0.0,0.118626,0.0,0.246991,0.544964,0.070694,0.065534,0.321412,0.370674,0.0
1,chr1:100034436-100035279,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0


### 6.2 RE-TG

In [52]:
LL_net.cis_reg(GRNdir,adata_RNA,adata_ATAC,genome,method,outdir)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 23/23 [01:35<00:00,  4.17s/it]


For each chromosome it combines the bulk RE-gene regulatory prior with a distance decay based on how far each peak is from the gene's TSS, producing a cre,gene,score table. Your single-cell data only contributes the peak and gene name sets for filtering — the actual scores come entirely from the bulk atlas and genomic distance.

```
cis_reg(method='baseline')
  └── for each chr:
        cis_reg_chr(...)
          ├── load_region(Region_overlap_{chr}.bed)
          │     → O_overlap, N_overlap, O_overlap_u, O_overlap_hg19_u
          ├── load_RE_TG(Primary_RE_TG_{chr}.txt)
          │     → sparse_S  (your peaks × genes, bulk prior scores)
          ├── filter genes present in adata_RNA.var
          ├── load_RE_TG_distance(RE_TG_distance_{chr}.txt)
          │     → sparse_dis  (your peaks × genes, distance decay weights)
          └── Score = sparse_S * sparse_dis
                → combined into cell_population_cis_regulatory.txt  (RE, gene, score)
```

In [53]:
ls LINGER_output

Region.bed                         cell_population_cis_regulatory.txt
Region_overlap_chr1.bed            chr10_cell_population_TF_RE_binding.txt
Region_overlap_chr10.bed           chr11_cell_population_TF_RE_binding.txt
Region_overlap_chr11.bed           chr12_cell_population_TF_RE_binding.txt
Region_overlap_chr12.bed           chr13_cell_population_TF_RE_binding.txt
Region_overlap_chr13.bed           chr14_cell_population_TF_RE_binding.txt
Region_overlap_chr14.bed           chr15_cell_population_TF_RE_binding.txt
Region_overlap_chr15.bed           chr16_cell_population_TF_RE_binding.txt
Region_overlap_chr16.bed           chr17_cell_population_TF_RE_binding.txt
Region_overlap_chr17.bed           chr18_cell_population_TF_RE_binding.txt
Region_overlap_chr18.bed           chr19_cell_population_TF_RE_binding.txt
Region_overlap_chr19.bed           chr1_cell_population_TF_RE_binding.txt
Region_overlap_chr2.bed            chr20_cell_population_TF_RE_binding.txt
Region_overlap_chr20.bed       

In [56]:
re_tg = pd.read_csv("LINGER_output/cell_population_cis_regulatory.txt", sep="\t")
re_tg.shape

(1528832, 3)

In [57]:
re_tg.head(2)

,chr1:100028489-100029404,PALMD,1.420250261127275e-08
0,chr1:100028489-100029404,RTCA,0.000005
1,chr1:100028489-100029404,SLC35A3,0.006177


### 6.3 TF-TG

In [54]:
LL_net.trans_reg(GRNdir,method,outdir,genome)

Generate trans-regulatory netowrk ...
Save trans-regulatory netowrk ...


The formula `Binding.T @ cis` multiplies TF-peak binding by peak-gene regulatory scores, effectively asking "how much does each TF bind to peaks that regulate each gene".

Then multiplied elementwise by TF_TG which is the direct bulk prior TF-gene score — acting as a filter that down-weights TF-gene links not supported by the bulk atlas.

```
trans_reg(method='baseline')
  ├── load cell_population_TF_RE_binding.txt  (output of TF_RE_binding)
  │     → Binding  (peaks × TFs)
  ├── load_cis(cell_population_cis_regulatory.txt)
  │     → cis  (peaks × genes, sparse matrix)
  ├── load_TF_TG(Primary_TF_TG_{chr}.txt for all chrs)
  │     → TF_TG  (genes × TFs, bulk prior direct TF-gene scores)
  └── S = Binding.T @ cis * TF_TG
        → cell_population_trans_regulatory.txt  (genes × TFs)
```

In [55]:
ls LINGER_output

Region.bed                         cell_population_cis_regulatory.txt
Region_overlap_chr1.bed            cell_population_trans_regulatory.txt
Region_overlap_chr10.bed           chr10_cell_population_TF_RE_binding.txt
Region_overlap_chr11.bed           chr11_cell_population_TF_RE_binding.txt
Region_overlap_chr12.bed           chr12_cell_population_TF_RE_binding.txt
Region_overlap_chr13.bed           chr13_cell_population_TF_RE_binding.txt
Region_overlap_chr14.bed           chr14_cell_population_TF_RE_binding.txt
Region_overlap_chr15.bed           chr15_cell_population_TF_RE_binding.txt
Region_overlap_chr16.bed           chr16_cell_population_TF_RE_binding.txt
Region_overlap_chr17.bed           chr17_cell_population_TF_RE_binding.txt
Region_overlap_chr18.bed           chr18_cell_population_TF_RE_binding.txt
Region_overlap_chr19.bed           chr19_cell_population_TF_RE_binding.txt
Region_overlap_chr2.bed            chr1_cell_population_TF_RE_binding.txt
Region_overlap_chr20.bed          

In [59]:
tf_tg = pd.read_csv("LINGER_output/cell_population_trans_regulatory.txt", sep="\t")
tf_tg.shape

(14842, 452)

In [60]:
tf_tg.head(2)   # TF x TG

,Unnamed: 0,PGR,BCL6,E2F4,MLXIP,XBP1,KLF1,CLOCK,MAF,FOXK2,...,ZBTB6,LMX1B,E2F5,GLIS1,IRF1,NRL,CTCF,HOMEZ,NPAS2,PLAGL1
0,PALMD,0.000005,0.000039,0.000008,0.000032,0.000021,0.000009,0.000015,0.000009,7.530552e-05,...,0.000008,0.000029,0.000023,0.000038,0.000048,0.000010,0.000034,0.000015,0.000011,0.000077
1,RTCA,0.000023,0.000029,0.000107,0.000059,0.000144,0.000294,0.000018,0.000005,8.715282e-07,...,0.000008,0.000036,0.000354,0.000010,0.000152,0.000062,0.000234,0.000127,0.000033,0.000013


## Conclusion

Baseline is a lookup-based approach. It takes your ATAC peaks, maps them to the bulk atlas coordinate system, and directly reads off precomputed regulatory scores. Your single-cell data contributes nothing to the scores themselves — only the peak and gene sets for filtering. 

"`baseline` is a naive method combining the prior GRNs and features from the single-cell data, offering a rapid approach"

```
TF_RE_binding  ←  Region_overlap_{chr}.bed  (from preprocess)
                   Primary_TF_RE_{chr}.txt   (from GRNdir)
                   adata_RNA.var             (gene names only)
      ↓
cell_population_TF_RE_binding.txt

cis_reg        ←  Region_overlap_{chr}.bed  (from preprocess)
                   Primary_RE_TG_{chr}.txt   (from GRNdir)
                   RE_TG_distance_{chr}.txt  (from GRNdir)
      ↓
cell_population_cis_regulatory.txt

trans_reg      ←  cell_population_TF_RE_binding.txt  (from TF_RE_binding)
                   cell_population_cis_regulatory.txt  (from cis_reg)
                   Primary_TF_TG_{chr}.txt             (from GRNdir)
      ↓
cell_population_trans_regulatory.txt

```

TF_RE_binding and cis_reg are independent of each other — both only depend on preprocess outputs and GRNdir.

trans_reg depends on both of them.